# Лабораторная работа № 4
## Объектно-ориентированное программирование. Принципы SOLID

**Среда выполнения:** Google Colab + GitLab

**Прикладная задача:** «Доменная модель интернет-магазина» — спроектировать объектную модель для управления товарами, клиентами, заказами и скидками, применяя инкапсуляцию, наследование, полиморфизм, магические методы и принципы SOLID.

**Автор:** <Гутьеррес Рейес О. С.>

**Группа:** <M26-534>

**Дата:** <23.10.2026>

---
## Часть I. Теория

### 1. Классы и объекты

**Класс** — это шаблон (чертёж) для создания объектов. **Объект** — экземпляр класса, обладающий своим состоянием и поведением. В Python классы определяются с помощью ключевого слова `class`.

```python
class Product:
    def __init__(self, name: str, price: float) -> None:
        self.name = name
        self.price = price
```

Метод `__init__` — это конструктор, который автоматически вызывается при создании объекта. Параметр `self` — это ссылка на сам экземпляр класса.

### 1.1. Инкапсуляция и свойства (`@property`)

Инкапсуляция — это скрытие внутреннего состояния объекта и предоставление контролируемого доступа к нему. В Python нет строгих модификаторов доступа (как `private` в C++), но есть соглашения:
*   `_name` — защищённый атрибут (protected). К нему не стоит обращаться снаружи.
*   `__name` — приватный атрибут (private). Python искажает его имя (name mangling), чтобы предотвратить случайный доступ.
*   `@property` — позволяет создать метод-геттер, который будет обращаться как обычный атрибут.
*   `@name.setter` — позволяет добавить логику (например, валидацию) при изменении значения.

```python
class Customer:
    def __init__(self, name: str, balance: float = 0.0) -> None:
        self.name = name
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Баланс не может быть отрицательным")
        self._balance = value
```

### 1.2. Магические методы

Магические методы (dunder methods, от double underscore) позволяют встроить объекты пользовательских классов в синтаксис языка Python.

| Метод | Вызов | Назначение |
|---|---|---|
| `__str__` | `str(obj)`, `print(obj)` | Читаемое представление для пользователя |
| `__repr__` | `repr(obj)` | Однозначное представление для отладки |
| `__eq__` | `obj1 == obj2` | Сравнение на равенство |
| `__add__` | `obj1 + obj2` | Сложение |
| `__len__` | `len(obj)` | Длина (количество элементов) |

```python
class OrderItem:
    def __init__(self, product: str, quantity: int) -> None:
        self.product = product
        self.quantity = quantity

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, OrderItem):
            return NotImplemented
        return self.product == other.product

    def __add__(self, other: "OrderItem") -> "OrderItem":
        if self.product != other.product:
            raise ValueError("Нельзя складывать разные товары")
        return OrderItem(self.product, self.quantity + other.quantity)
```

### 1.3. Наследование и полиморфизм

**Наследование** позволяет создавать новый класс на основе существующего, переиспользуя его код. Дочерний класс вызывает `super().__init__()` для инициализации родительской части.

**Полиморфизм** — это способность объектов с одинаковым интерфейсом (например, наследников одного базового класса) вести себя по-разному. Вызов метода приводит к выполнению кода, соответствующего конкретному классу объекта.

```python
class Customer:
    def get_discount(self) -> float:
        return 0.0

class VipCustomer(Customer):
    def get_discount(self) -> float:
        return 0.10
```

### 1.4. Абстрактные классы (ABC)

**Абстрактный класс** — это класс, который нельзя инстанцировать (создать его объект). Он служит контрактом для наследников. Если наследник не реализует все абстрактные методы, создать его объект тоже не получится.

```python
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        ...
```

### 1.5. Принципы SOLID

SOLID — это пять принципов объектно-ориентированного проектирования, делающих систему гибкой и поддерживаемой.

| Принцип | Расшифровка | Суть |
|---|---|---|
| **SRP** | Single Responsibility | У класса должна быть только одна причина для изменения. |
| **OCP** | Open/Closed | Классы открыты для расширения, но закрыты для модификации. |
| **LSP** | Liskov Substitution | Наследник должен без ошибок подставляться вместо родителя. |
| **ISP** | Interface Segregation | Лучше много специализированных интерфейсов, чем один огромный. |
| **DIP** | Dependency Inversion | Зависимость от абстракций, а не от конкретных реализаций. |

---
## Примеры

В качестве прикладной задачи рассмотрим доменную модель интернет-магазина. У нас есть товары, клиенты (с балансом), заказы (состоящие из позиций) и скидки. В будущем заказы нужно будет сохранять в базу данных.

### Пример 1. Инкапсуляция и валидация данных

In [5]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        self.name = name
        self.price = price
        self.stock = stock

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if value < 0:
            raise ValueError("Цена не может быть отрицательной")
        self._price = value

# Проверка
p = Product("Ноутбук", 75000.0, 10)
print(p.price)
try:
    p.price = -100
except ValueError as e:
    print("Ошибка:", e)

75000.0
Ошибка: Цена не может быть отрицательной


### Пример 2. Наследование и `super()`

In [3]:
class Customer:
    def __init__(self, name: str, balance: float = 0.0):
        self.name = name
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма списания должна быть положительной")
        if amount > self._balance:
            raise ValueError(
                f"Недостаточно средств: запрошено {amount}, доступно {self._balance}"
            )
        self._balance -= amount


# Проверка
c = Customer("Иван", 1000.0)
print(c.name, c.balance)   # Иван 1000.0

c.deposit(500)
print(c.balance)           # 1500.0

c.withdraw(200)
print(c.balance)           # 1300.0

try:
    c.withdraw(5000)
except ValueError as e:
    print("Ошибка:", e)    # Ошибка: Недостаточно средств: ...

try:
    c.balance = 999        # свойство только для чтения
except AttributeError as e:
    print("Ошибка:", e)    # property 'balance' of 'Customer' object has no setter

Иван 1000.0
1500.0
1300.0
Ошибка: Недостаточно средств: запрошено 5000, доступно 1300.0
Ошибка: property 'balance' of 'Customer' object has no setter


### Пример 3. Композиция и магические методы

In [6]:
from __future__ import annotations


class OrderItem:
    def __init__(self, product: Product, quantity: int) -> None:
        # quantity <= 0 ломает subtotal, поэтому валидируем сразу
        if quantity <= 0:
            raise ValueError("Количество должно быть положительным")
        self.product = product
        self.quantity = quantity

    @property
    def subtotal(self) -> float:
        return self.product.price * self.quantity

    # без repr отладка списка позиций бесполезна
    def __repr__(self) -> str:
        return f"OrderItem({self.product.name!r}, qty={self.quantity})"


class Order:
    def __init__(self, customer: Customer) -> None:
        self.customer = customer
        self.items: list[OrderItem] = []
        self.discount = None  # любой объект с apply(base) -> float

    def add_item(self, item: OrderItem) -> None:
        # лучше упасть здесь, чем потом в total() с непонятной ошибкой
        if not isinstance(item, OrderItem):
            raise TypeError("Ожидается объект OrderItem")
        self.items.append(item)

    def total(self) -> float:
        base = sum(i.subtotal for i in self.items)
        # is not None, а не truthy: скидка 0% не должна отключать применение
        if self.discount is not None:
            base = self.discount.apply(base)
        # float любит давать 0.30000000000000004, округляем
        return round(base, 2)

    def __len__(self) -> int:
        return len(self.items)

    def __str__(self) -> str:
        # len(self) вместо len(self.items) — переиспользуем __len__
        return (
            f"Заказ для {self.customer.name}. "
            f"Позиций: {len(self)}. Итого: {self.total():.2f} руб."
        )

    # для отладки: print([order]) должен быть читаемым
    def __repr__(self) -> str:
        return (
            f"Order(customer={self.customer.name!r}, "
            f"items={len(self)}, total={self.total()!r})"
        )


# пример скидки, чтобы зафиксировать контракт discount
class PercentDiscount:
    def __init__(self, percent: float) -> None:
        if not 0 <= percent <= 100:
            raise ValueError("Процент должен быть в диапазоне 0..100")
        self.percent = percent

    def apply(self, base: float) -> float:
        return base * (1 - self.percent / 100)


# проверка
p1 = Product("Ноутбук", 75999.99, 12)
p2 = Product("Мышь", 1999.50, 100)

o = Order(Customer("Иван", 100000.0))
o.add_item(OrderItem(p1, 1))
o.add_item(OrderItem(p2, 2))
print(o)                 # Заказ для Иван. Позиций: 2. Итого: 79999.00 руб.
print(len(o))            # 2

o.discount = PercentDiscount(10)
print(o)                 # ... Итого: 71999.10 руб.
print(repr(o))

Заказ для Иван. Позиций: 2. Итого: 79998.99 руб.
2
Заказ для Иван. Позиций: 2. Итого: 71999.09 руб.
Order(customer='Иван', items=2, total=71999.09)


### Пример 4. Полиморфизм и принцип OCP (Open/Closed)

In [7]:
from abc import ABC, abstractmethod


class Discount(ABC):
    # контракт: любая скидка умеет применить себя к сумме
    @abstractmethod
    def apply(self, total: float) -> float: ...


class PercentageDiscount(Discount):
    def __init__(self, percent: float) -> None:
        # процент вне 0..100 — это уже не скидка, а баг
        if not 0 <= percent <= 100:
            raise ValueError("Процент должен быть в диапазоне 0..100")
        self.percent = percent

    def apply(self, total: float) -> float:
        return total * (1 - self.percent / 100)

    # без repr в отладке не видно, какая скидка сработала
    def __repr__(self) -> str:
        return f"PercentageDiscount({self.percent}%)"


class FixedDiscount(Discount):
    def __init__(self, amount: float) -> None:
        # отрицательная фиксированная скидка увеличивала бы чек
        if amount < 0:
            raise ValueError("Сумма скидки не может быть отрицательной")
        self.amount = amount

    def apply(self, total: float) -> float:
        # max(0, ...) — чтобы итог не ушёл в минус
        return max(0, total - self.amount)

    def __repr__(self) -> str:
        return f"FixedDiscount({self.amount})"


# Принцип OCP здесь работает так: чтобы добавить новую скидку,
# мы создаём нового наследника Discount, не меняя код класса Order.

### Пример 5. Принцип DIP (Dependency Inversion)

In [ ]:
import uuid

class OrderRepository(ABC):
    @abstractmethod
    def save(self, order: Order) -> str: ...

class InMemoryOrderRepository(OrderRepository):
    def __init__(self) -> None:
        self._storage = {}

    def save(self, order: Order) -> str:
        order_id = str(uuid.uuid4())
        self._storage[order_id] = order
        return order_id

class OrderService:
    # Сервис зависит от абстракции OrderRepository, а не от конкретной реализации
    def __init__(self, repository: OrderRepository) -> None:
        self.repository = repository

    def place_order(self, order: Order) -> str:
        return self.repository.save(order)

# Проверка сквозного сценария
p1 = Product("Книга", 1000.0, 5)
c1 = Customer("Иван", 5000.0)
order = Order(c1)
order.add_item(OrderItem(p1, 2))

repo = InMemoryOrderRepository()
service = OrderService(repo)
order_id = service.place_order(order)
print(order)
print("Заказ сохранен с ID:", order_id)

Заказ для Иван. Позиций: 1. Итого: 2000.0 руб.
Заказ сохранен с ID: 69707b35-4fda-4f2b-a2ff-01d0d82f923e


---
## Часть II. Задания

### Задания на освоение концепции (1–10)

**Задание 1. Базовый класс и магический метод `__str__`.**
Создайте класс `Product` с полями `name: str`, `price: float`, `stock: int`. Реализуйте `__str__` для красивого вывода и `__repr__` для отладки. Проверьте работу методов через `print()` и `repr()`.

In [1]:
    
class Product:
    def __init__(self, name: str, price: float, stock: int):
        self.name = name
        self.price = price
        self.stock = stock

    def __str__(self):
        return f"Товар: {self.name}, цена: {self.price:.2f} руб., на складе: {self.stock} шт."

    def __repr__(self):
        return f"Product(name={self.name!r}, price={self.price!r}, stock={self.stock!r})"


# Проверка
p = Product("Ноутбук", 75999.99, 12)
print(p)        # вызывает __str__
print(repr(p))  # вызывает __repr__

Товар: Ноутбук, цена: 75999.99 руб., на складе: 12 шт.
Product(name='Ноутбук', price=75999.99, stock=12)


**Задание 2. Инкапсуляция и свойства (`@property`).**
Создайте класс `Customer` с полями `name: str` и `_balance: float`. Реализуйте `balance` как свойство только для чтения. Добавьте методы `deposit(amount)` и `withdraw(amount)`. При попытке списать больше, чем есть, поднимайте `ValueError`.

In [ ]:
class Customer:
    def __init__(self, name: str, balance: float = 0.0) -> None:
        self.name = name
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> None:
        # Ваш код
        ...

    def withdraw(self, amount: float) -> None:
        # Ваш код (должен выбрасывать ValueError)
        ...

**Задание 3. Магические методы сравнения и сложения.**
Создайте класс `OrderItem` (товар + количество). Реализуйте `__eq__` для сравнения позиций по товару и `__add__` для сложения двух позиций одного товара (увеличение количества). Проверьте: `OrderItem("Мышь", 1) + OrderItem("Мышь", 2)`.

In [ ]:
class OrderItem:
    def __init__(self, product: str, quantity: int) -> None:
        self.product = product
        self.quantity = quantity

    def __eq__(self, other: object) -> bool:
        # Ваш код
        ...

    def __add__(self, other: "OrderItem") -> "OrderItem":
        # Ваш код
        ...

**Задание 4. Наследование и `super()`.**
Создайте класс `VipCustomer(Customer)`, который при создании получает бонусный баланс, а метод `get_discount` возвращает 10%. Используйте `super()` для вызова конструктора родителя.

In [ ]:
class VipCustomer(Customer):
    def __init__(self, name: str, balance: float = 0.0, bonus: float = 500.0) -> None:
        # Ваш код
        ...

    def get_discount(self) -> float:
        return 0.10

**Задание 5. `@classmethod` и `@staticmethod`.**
Добавьте в `Product`:
1. `@classmethod` `from_dict(cls, data: dict) -> Product` для создания товара из словаря.
2. `@staticmethod` `is_valid_price(price: float) -> bool` для проверки, что цена положительная.

In [ ]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        self.name = name
        self.price = price
        self.stock = stock

    @classmethod
    def from_dict(cls, data: dict) -> "Product":
        # Ваш код
        ...

    @staticmethod
    def is_valid_price(price: float) -> bool:
        # Ваш код
        ...

**Задание 6. Абстрактный базовый класс (ABC).**
Создайте `Discount(ABC)` с абстрактным методом `apply(total: float) -> float`. Убедитесь, что попытка создать экземпляр `Discount()` вызывает `TypeError`.

In [ ]:
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        ...

**Задание 7. Полиморфизм.**
Реализуйте наследников `Discount`: `PercentageDiscount(percent)` и `FixedDiscount(amount)`. Убедитесь, что метод `apply` возвращает новую сумму. Проверьте на сумме 1000 руб.

In [ ]:
class PercentageDiscount(Discount):
    def __init__(self, percent: float) -> None:
        self.percent = percent

    def apply(self, total: float) -> float:
        # Ваш код
        ...

class FixedDiscount(Discount):
    def __init__(self, amount: float) -> None:
        self.amount = amount

    def apply(self, total: float) -> float:
        # Ваш код
        ...

**Задание 8. Композиция.**
Создайте класс `Order`, который содержит `customer: Customer`, `items: list[OrderItem]`, `discount: Discount | None`. Метод `total()` возвращает сумму с учётом скидки.

In [ ]:
class Order:
    def __init__(self, customer: Customer) -> None:
        # Ваш код
        ...

    def add_item(self, item: OrderItem) -> None:
        # Ваш код
        ...

    def total(self) -> float:
        # Ваш код
        ...

**Задание 9. Dependency Injection.**
Создайте класс `OrderService`, который принимает в конструкторе `repository` (хранилище заказов). Сервис не должен знать, как именно хранятся заказы — только интерфейс.

In [ ]:
class OrderRepository(ABC):
    @abstractmethod
    def save(self, order: Order) -> str: ...

class OrderService:
    def __init__(self, repository: OrderRepository) -> None:
        self.repository = repository

    def place_order(self, order: Order) -> str:
        # Ваш код
        ...

**Задание 10. Принципы SOLID в действии.**
В комментариях к коду (из заданий 8 и 9) расставьте, где применяется каждый принцип: SRP, OCP, LSP, DIP.

In [ ]:
# SRP: Order отвечает за бизнес-логику заказа, OrderService — за оркестрацию, OrderRepository — за хранение
# OCP: добавление новой скидки не требует изменения Order
# LSP: любой наследник Discount подставляется в Order без ошибок
# DIP: OrderService зависит от абстракции OrderRepository

---
### Задания по прикладной задаче (11–20)

**Прикладная задача:** «Доменная модель интернет-магазина».
Требуется собрать систему, в которой клиенты могут делать заказы, товары имеют остатки на складе, к заказам применяются скидки, а сами заказы сохраняются в хранилище.

**Задание 11. Настройка окружения и валидация Product**
Инициализируйте папку `lab4` в репозитории `python-labs`. Создайте ноутбук. Реализуйте класс `Product` с валидацией в `@property` (цена и остаток не могут быть отрицательными).

In [ ]:
!git clone https://gitlab.com/<логин>/python-labs.git
%cd python-labs
!mkdir -p lab4
%cd lab4
!git config --global user.name "Ваше Имя"
!git config --global user.email "ваш@email.com"

/bin/bash: line 1: логин: No such file or directory
[Errno 2] No such file or directory: 'python-labs'
/content
/content/lab4


In [ ]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        self.name = name
        self.price = price
        self.stock = stock

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        # Ваш код
        ...

    @property
    def stock(self) -> int:
        return self._stock

    @stock.setter
    def stock(self, value: int) -> None:
        # Ваш код
        ...

**Задание 12. Иерархия клиентов**
Реализуйте класс `Customer` (с балансом и методами `deposit/withdraw`) и `VipCustomer` (с бонусом при создании и скидкой 10%). Обрабатывайте ошибку при недостатке средств.

In [ ]:
class Customer:
    # Ваш код
    ...

class VipCustomer(Customer):
    # Ваш код
    ...

**Задание 13. Позиция заказа и сам заказ**
Реализуйте `OrderItem` (товар, количество, подсчёт `subtotal`). Реализуйте класс `Order`, поддерживающий добавление позиций, подсчёт суммы, применение скидки, вывод через `__str__` и подсчёт количества позиций через `__len__`.

In [ ]:
class OrderItem:
    # Ваш код
    ...

class Order:
    # Ваш код
    ...

**Задание 14. Иерархия скидок**
Создайте `Discount(ABC)`, `PercentageDiscount`, `FixedDiscount` и `NoDiscount`. Убедитесь, что `Order.total()` корректно работает с любой из них.

In [ ]:
# Ваш код
...

**Задание 15. Абстрактный репозиторий (DIP)**
Создайте `OrderRepository(ABC)` с методами `save(order)`, `get_by_id(order_id)`, `get_all()`.

In [ ]:
# Ваш код
...

**Задание 16. Реализация in-memory хранилища**
Создайте `InMemoryOrderRepository(OrderRepository)`, который хранит заказы в словаре. Для генерации ID используйте модуль `uuid`.

In [ ]:
# Ваш код
...

**Задание 17. Сервис заказов**
Реализуйте `OrderService`, который принимает репозиторий и предоставляет метод `place_order(order)` для сохранения.

In [ ]:
# Ваш код
...

**Задание 18. Сборка сценария (Main)**
Напишите функцию `main()`, которая:
1. Создаёт 2-3 товара.
2. Создаёт клиента и пополняет его баланс.
3. Формирует заказ с несколькими позициями.
4. Применяет скидку.
5. Сохраняет заказ через `OrderService`.
6. Печатает чек (вывод заказа) и сообщает ID сохранённого заказа.

In [ ]:
def main() -> None:
    # Ваш код
    ...

main()

**Задание 19. Обработка ошибок в домене**
Добавьте проверку: если товара не хватает на складе (`product.stock < quantity`), при добавлении в заказ должно выбрасываться исключение `ValueError`. Обработайте его в `main()`.

In [ ]:
# Ваш код
...

**Задание 20. Комплексный отчёт и сдача**
1. Добавьте docstring ко всем классам и методам.
2. Проверьте код линтером (PEP 8).
3. Создайте `README.md`: опишите архитектуру классов и обоснуйте применение принципов SOLID (особенно DIP и OCP).
4. Сделайте коммит и `git push` в GitLab.

In [ ]:
%%writefile README.md
# Лабораторная работа № 4

## Тема
Объектно-ориентированное программирование. Принципы SOLID.

## Архитектура
- `Product` — товар, инкапсуляция цены и остатка.
- `Customer` — клиент с балансом, `VipCustomer` — скидка 10%.
- `OrderItem` — позиция заказа, магические методы `__eq__`, `__add__`.
- `Order` — заказ, композиция позиций и скидки.
- `Discount` (ABC) — абстракция скидки, реализации `PercentageDiscount`, `FixedDiscount`.
- `OrderRepository` (ABC) — интерфейс хранилища, `InMemoryOrderRepository` — реализация.
- `OrderService` — оркестратор, зависит от `OrderRepository` (DIP).

## Принципы SOLID
- **SRP:** `Order` считает сумму, `Discount` — скидку, `Repository` — хранит.
- **OCP:** новая скидка реализуется без изменения `Order`.
- **LSP:** любой наследник `Discount` подставляется в `Order`.
- **DIP:** `OrderService` зависит от абстракции, а не от `InMemoryOrderRepository`.

## Ссылка на репозиторий
https://gitlab.com/<логин>/python-labs

Writing README.md


In [ ]:
!git add .
!git commit -m "ЛР4: ООП, SOLID, доменная модель магазина"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


## Контрольные вопросы

1. Чем `__str__` отличается от `__repr__`?
2. Зачем нужен `@property`, если можно просто сделать атрибут публичным?
3. Что такое `@classmethod` и `@staticmethod`? В чём разница?
4. Как `super()` работает в цепочке наследования?
5. Что такое абстрактный класс и зачем нужны абстрактные методы?
6. Как перегрузка операторов (магические методы) помогает сделать код читаемее?
7. Объясните принцип Open/Closed на примере иерархии скидок.
8. Что такое Dependency Injection и почему это лучше, чем создавать объекты внутри класса?
9. Как принцип подстановки Лисков (LSP) связан с исключениями в наследниках?
10. Почему считается, что наследование увеличивает связанность кода, и как композиция помогает это исправить?

## Дополнительно

1. Оформите выполненные задания в виде Jupyter-ноутбука `lab4_oop.ipynb`.
2. Убедитесь, что ноутбук выполняется сверху вниз без ошибок (Kernel → Restart & Run All).
3. В репозитории `python-labs` создайте папку `lab4` и добавьте туда ноутбук и `README.md`.
4. Создайте `.gitignore` (исключите `__pycache__/`, `.ipynb_checkpoints/`).
5. Зафиксируйте изменения:
```bash
git add .
git commit -m "ЛР4: ООП, SOLID, доменная модель магазина"
git push origin main
```
6. В конце `README.md` добавьте раздел «Ссылка на репозиторий» и укажите URL.
7. Прикрепите ссылку и ноутбук в системе дистанционного обучения.